## Preprocessing
#### بجهز الداتا قبل تدريب الموديل عشان احسن الاداء واقلل المشاكل

In [31]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression

In [32]:
df = pd.read_csv('final_internship_data.csv')
df.shape

(3073, 26)

### Missing value
##### بشوف الاعمده اللي فيها قيم ناقصه وبعالجها بالطريقه المناسبه

In [33]:
missing_percentage = df.isnull().mean() * 100
missing_percentage = missing_percentage[missing_percentage > 0].sort_values(ascending=False)
print(missing_percentage)

Car Condition        0.032541
day                  0.032541
distance             0.032541
nyc_dist             0.032541
sol_dist             0.032541
lga_dist             0.032541
ewr_dist             0.032541
jfk_dist             0.032541
year                 0.032541
weekday              0.032541
month                0.032541
hour                 0.032541
Weather              0.032541
passenger_count      0.032541
dropoff_latitude     0.032541
dropoff_longitude    0.032541
pickup_latitude      0.032541
pickup_longitude     0.032541
pickup_datetime      0.032541
fare_amount          0.032541
key                  0.032541
Traffic Condition    0.032541
bearing              0.032541
dtype: float64


### Duplicate
##### بشوف لو في بيانات متكرره ولو في اشيلها

In [34]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()

Duplicate rows: 0


In [35]:
df = df.dropna(how='all', subset=[c for c in df.columns if c not in ['User ID','User Name','Driver Name']])
df = df.drop(columns=['User ID', 'User Name', 'Driver Name', 'key'])
df.shape

(3072, 22)

### Feature Engineering
##### جديدة عشان تساعد الموديل يطلع نتايج احسن(Features)هضيف

In [36]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df = df.drop(columns=['pickup_datetime', 'hour'])
df.columns

Index(['Car Condition', 'Weather', 'Traffic Condition', 'fare_amount',
       'pickup_longitude', 'pickup_latitude', 'dropoff_longitude',
       'dropoff_latitude', 'passenger_count', 'day', 'month', 'weekday',
       'year', 'jfk_dist', 'ewr_dist', 'lga_dist', 'sol_dist', 'nyc_dist',
       'distance', 'bearing', 'hour_sin', 'hour_cos'],
      dtype='object')

In [37]:
df = df[df['fare_amount'] > 0]
df = df[df['passenger_count'] > 0]
df = df[df['distance'] > 0]
df = df[df['distance'] <= 100]
df.shape

(2975, 22)

### Outliers
##### هراجع القيم الشاذة واقرر اتعامل معاها ازاي 

In [38]:
for col in ['fare_amount', 'distance']:
    cap_value = df[col].quantile(0.99)
    df[col] = np.where(df[col] > cap_value, cap_value, df[col])
df[['fare_amount', 'distance']].describe()

,fare_amount,distance
count,2975.000000,2975.000000
mean,11.221277,3.341572
std,8.717349,3.460468
min,0.010000,0.000279
25%,6.000000,1.273928
50%,8.500000,2.188903
75%,12.900000,4.032141
max,52.000000,20.233018


### Feature Selection
##### هشيل الاعمده اللي ملهاش تأثير كبير علي الموديل

In [39]:
df = df.drop(columns=['ewr_dist', 'lga_dist', 'sol_dist', 'nyc_dist'])
df.shape

(2975, 18)

### Train Test Split
##### هقسم الداتا لتريب واختبار عشان اقيم الموديل 

In [40]:
X = df.drop(columns=['fare_amount'])
y = df['fare_amount']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)
X_train.shape, X_test.shape

((2380, 17), (595, 17))

### Encoding
##### هحول الاعمده النصيه لارقام عشان اعرف اتعامل معاها

In [41]:
car_condition_order = ['Bad', 'Good', 'Very Good', 'Excellent']
traffic_condition_order = ['Flow Traffic', 'Dense Traffic', 'Congested Traffic']
ordinal_cols = ['Car Condition', 'Traffic Condition']
onehot_cols = ['Weather']
numeric_cols = [c for c in X.columns if c not in ordinal_cols + onehot_cols]
preprocessor = ColumnTransformer(transformers=[
    ('ordinal_car', OrdinalEncoder(categories=[car_condition_order]), ['Car Condition']),
    ('ordinal_traffic', OrdinalEncoder(categories=[traffic_condition_order]), ['Traffic Condition']),
    ('onehot', OneHotEncoder(handle_unknown='ignore'), onehot_cols),
    ('scale', StandardScaler(), numeric_cols),
])
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
X_train_processed.shape, X_test_processed.shape

((2380, 21), (595, 21))

### Baseline Model
##### موديل بسيط عشان اشوف الاداء

In [42]:
baseline_model = LinearRegression()
cv_scores = cross_val_score(baseline_model, X_train_processed, y_train_log, cv=5, scoring='neg_root_mean_squared_error')
-cv_scores, (-cv_scores).mean()

(array([0.31653515, 0.36493678, 0.30292616, 2.94147388, 1.42524548]),
 np.float64(1.0702234882095827))

In [43]:
X_train[['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'bearing']].describe()

,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,bearing
count,2380.000000,2380.000000,2380.000000,2380.000000,2380.000000
mean,-1.289739,0.710097,-1.289723,0.710106,0.327962
std,0.048832,0.043554,0.048836,0.043553,1.823767
min,-1.299192,-1.291316,-1.294790,-1.291166,-3.130088
25%,-1.291418,0.710965,-1.291396,0.710963,-0.863187
50%,-1.291233,0.711273,-1.291206,0.711300,-0.134723
75%,-1.290995,0.711524,-1.290942,0.711536,2.278407
max,0.711249,0.721975,0.711376,0.713779,3.140999


In [44]:
lat_lower = X_train['pickup_latitude'].quantile(0.01)
lat_upper = X_train['pickup_latitude'].quantile(0.99)
lon_lower = X_train['pickup_longitude'].quantile(0.01)
lon_upper = X_train['pickup_longitude'].quantile(0.99)
mask = (
    (X_train['pickup_latitude'] < lat_lower) | (X_train['pickup_latitude'] > lat_upper) |
    (X_train['pickup_longitude'] < lon_lower) | (X_train['pickup_longitude'] > lon_upper)
)
print(mask.sum())
X_train[mask][['pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude']]

79


,pickup_latitude,pickup_longitude,dropoff_latitude,dropoff_longitude
2287,0.710606,-1.291817,0.711954,-1.290935
3058,0.709383,-1.287738,0.709864,-1.290566
2283,0.709326,-1.287823,0.708216,-1.290995
2794,0.709365,-1.287880,0.710065,-1.290490
2735,0.710505,-1.291842,0.711250,-1.291408
...,...,...,...,...
1645,0.709382,-1.287747,0.710235,-1.290198
1,0.710546,-1.291824,0.711780,-1.291182
1701,0.710517,-1.291811,0.711363,-1.291356
809,0.709384,-1.287743,0.711013,-1.291078


### Cross Validation
##### هقيم الموديل بطريقه ادق

In [45]:
from sklearn.linear_model import Ridge
baseline_model = Ridge(alpha=1.0)
cv_scores = cross_val_score(baseline_model, X_train_processed, y_train_log, cv=5, scoring='neg_root_mean_squared_error')
-cv_scores, (-cv_scores).mean()

(array([0.31632397, 0.35665294, 0.30326049, 1.07052694, 0.60368557]),
 np.float64(0.5300899833630379))

##### هقارن النتائج واشوف الموديل بعد التحسين

In [46]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
baseline_model.fit(X_train_processed, y_train_log)
y_pred_log = baseline_model.predict(X_test_processed)
y_pred = np.expm1(y_pred_log)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 3.0206077598961087
RMSE: 6.740431906552697
R2: 0.43491984210883716


### Hyperparameter
##### هجرب افضل اعدادات للموديل عشان اوصل لاحسن نتيجه

In [47]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}
rf = RandomForestRegressor(random_state=42)
rf_search = RandomizedSearchCV(
    rf, param_distributions=param_dist,
    n_iter=20, cv=5, scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1
)
rf_search.fit(X_train_processed, y_train_log)
print("Best params:", rf_search.best_params_)
print("Best CV RMSE (log scale):", -rf_search.best_score_)

Best params: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}
Best CV RMSE (log scale): 0.26211923365256407


In [48]:
best_rf = rf_search.best_estimator_
y_pred_rf_log = best_rf.predict(X_test_processed)
y_pred_rf = np.expm1(y_pred_rf_log)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)
print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R2:", r2_rf)

MAE: 2.2331593665587697
RMSE: 4.2326376856495305
R2: 0.777178653040158


### Piprline
##### هجمع خطوات تجهيز الداتا مع الموديل في(Pipeline)واحده عشان انظم

In [49]:
from sklearn.pipeline import Pipeline
import joblib
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', best_rf)
])
final_pipeline.fit(X_train, y_train_log)
joblib.dump(final_pipeline, 'final_pipeline.pkl')
print("saved")

saved
